In [1]:
# Install necessary packages
%pip install langchain sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.


In [47]:
import os
import json
from datetime import datetime
from typing import List, Dict
from langchain.docstore.document import Document
from langchain.text_splitter import CharacterTextSplitter

In [51]:
folder_paths = ["../data/cleaned_companydata"]


In [55]:
# Function to Load All JSON Files from the Specified Folders
def load_json_files(folder_paths: List[str]) -> List[Dict]:
    """Loads all JSON files from the specified folders and combines them into a list."""
    data = []
    for folder_path in folder_paths:
        for file in os.listdir(folder_path):
            if file.endswith(".json"):
                file_path = os.path.join(folder_path, file)
                with open(file_path, "r", encoding="utf-8") as f:
                    data.append(json.load(f))  # Append dictionary instead of extending list
    return data

In [59]:

# Call the function to load data
json_data = load_json_files(folder_paths)

# Print first few records to verify
print("First few records from the JSON files:", json_data[:5])

First few records from the JSON files: [{'2019-01-07': {'price': {'max': 227, 'min': 227, 'close': 227, 'prevClose': 227, 'diff': 0}, 'numTrans': 1, 'tradedShares': 2, 'amount': 454}, '2019-01-08': {'price': {'max': 238, 'min': 227, 'close': 238, 'prevClose': 227, 'diff': 11}, 'numTrans': 1, 'tradedShares': 10, 'amount': 2380}, '2019-01-31': {'price': {'max': 258, 'min': 238, 'close': 258, 'prevClose': 238, 'diff': 20}, 'numTrans': 3, 'tradedShares': 30, 'amount': 7600}, '2019-02-05': {'price': {'max': 270, 'min': 258, 'close': 270, 'prevClose': 258, 'diff': 12}, 'numTrans': 1, 'tradedShares': 10, 'amount': 2700}, '2019-02-06': {'price': {'max': 283, 'min': 270, 'close': 283, 'prevClose': 270, 'diff': 13}, 'numTrans': 1, 'tradedShares': 10, 'amount': 2830}, '2019-02-11': {'price': {'max': 311, 'min': 283, 'close': 311, 'prevClose': 283, 'diff': 28}, 'numTrans': 5, 'tradedShares': 50, 'amount': 15290}, '2019-02-12': {'price': {'max': 326, 'min': 311, 'close': 326, 'prevClose': 311, 'dif

In [61]:
# Initialize an empty dictionary to hold all the data
all_json_data = {}

# Load data from each JSON file into the all_json_data dictionary
for json_file in loaded_files:
	file_path = os.path.join(folder_paths,json_file)
	
	with open(file_path, 'r') as file:
		data = json.load(file)
		
		# Store the data with the file name as a key or include it in the metadata
		all_json_data[json_file] = data

# Print the loaded data with the filenames included
print("Data loaded from JSON files:", all_json_data)

TypeError: expected str, bytes or os.PathLike object, not list

In [41]:
# Function to Standardize Date Format
def normalize_date(date_str: str, date_format: str = "%Y-%m-%d") -> str:
    """Converts various date formats to a standard format."""
    try:
        return datetime.strptime(date_str, date_format).strftime("%Y-%m-%d")
    except ValueError:
        return None

In [35]:
def process_json_data(json_data: List[Dict], date_key: str = "date") -> str:
    cleaned_pages = []
    for record in json_data:
        if isinstance(record, dict):
            if date_key in record:
                record[date_key] = normalize_date(record[date_key])
            cleaned_pages.append(record.get("content", ""))
    print("First processed page:", cleaned_pages[0] if cleaned_pages else "No processed data")
    return "\n".join(cleaned_pages)

In [62]:
# Function to Process Stock Data
def process_stock_json(json_data: Dict) -> List[Document]:
    """
    Processes stock data where dates are keys.
    Converts price and transaction details into Document objects for FAISS.
    """
    documents = []
    
    for date, data in json_data.items():
        # Extract price details
        price_data = data.get("price", {})
        open_price = price_data.get("open", "N/A")
        close_price = price_data.get("close", "N/A")
        max_price = price_data.get("max", "N/A")
        min_price = price_data.get("min", "N/A")
        prev_close = price_data.get("prevClose", "N/A")
        price_diff = price_data.get("diff", "N/A")

        # Extract trading details
        num_transactions = data.get("numTrans", "N/A")
        traded_shares = data.get("tradedShares", "N/A")
        trade_amount = data.get("amount", "N/A")

        # Format as readable text
        content = f"""
        Date: {date}
        Open Price: {open_price}
        Max Price: {max_price}
        Min Price: {min_price}
        Close Price: {close_price}
        Previous Close: {prev_close}
        Price Difference: {price_diff}
        Number of Transactions: {num_transactions}
        Traded Shares: {traded_shares}
        Total Trade Amount: {trade_amount}
        """

        # Create a Document object with metadata
        doc = Document(
            page_content=content.strip(),
            metadata={
                "date": date,
                "open_price": open_price,
                "close_price": close_price,
                "max_price": max_price,
                "min_price": min_price,
                "num_transactions": num_transactions,
                "traded_shares": traded_shares
            }
        )
        documents.append(doc)

    print("First processed document:", documents[0] if documents else "No processed data")
    return documents

In [46]:
def chunk_and_embed(json_data: Dict):
    """
    Splits JSON data by treating each date's record as a separate chunk.
    """
    chunks = []

    for date, data in json_data.items():
        # Convert each date's data into a formatted string
        content = f"""
        Date: {date}
        Open Price: {data["price"]["open"]}
        Max Price: {data["price"]["max"]}
        Min Price: {data["price"]["min"]}
        Close Price: {data["price"]["close"]}
        Previous Close: {data["price"]["prevClose"]}
        Price Difference: {data["price"]["diff"]}
        Number of Transactions: {data["numTrans"]}
        Traded Shares: {data["tradedShares"]}
        Total Trade Amount: {data["amount"]}
        """

        # Create a Document object for each date
        doc = Document(page_content=content.strip(), metadata={"date": date})
        chunks.append(doc)

    print("First chunk:", chunks[0] if chunks else "No chunks generated")
    return chunks

# Call the function
chunks = chunk_and_embed(all_json_data)


AttributeError: 'list' object has no attribute 'items'

In [19]:
from langchain.schema import Document
from typing import List, Dict

def process_stock_json(json_data: Dict) -> List[Document]:
    """
    Processes stock data where dates are keys.
    Converts price and transaction details into Document objects for FAISS.
    """
    documents = []

    for date, data in json_data.items():
        # Extract price details
        price_data = data.get("price", {})
        open_price = price_data.get("open", "N/A")
        close_price = price_data.get("close", "N/A")
        max_price = price_data.get("max", "N/A")
        min_price = price_data.get("min", "N/A")
        prev_close = price_data.get("prevClose", "N/A")
        price_diff = price_data.get("diff", "N/A")

        # Extract trading details
        num_transactions = data.get("numTrans", "N/A")
        traded_shares = data.get("tradedShares", "N/A")
        trade_amount = data.get("amount", "N/A")

        # Format as readable text
        content = f"""
        Date: {date}
        Open Price: {open_price}
        Max Price: {max_price}
        Min Price: {min_price}
        Close Price: {close_price}
        Previous Close: {prev_close}
        Price Difference: {price_diff}
        Number of Transactions: {num_transactions}
        Traded Shares: {traded_shares}
        Total Trade Amount: {trade_amount}
        """

        # Create a Document object with metadata
        doc = Document(
            page_content=content.strip(),
            metadata={
                "date": date,
                "open_price": open_price,
                "close_price": close_price,
                "max_price": max_price,
                "min_price": min_price,
                "num_transactions": num_transactions,
                "traded_shares": traded_shares
            }
        )
        documents.append(doc)

    return documents

# Sample stock JSON data (Replace with your actual data)
json_data = {
    "2024-10-17": {
        "price": {"open": 1100, "max": 1100, "min": 1061, "close": 1079, "prevClose": 1080, "diff": -1},
        "numTrans": 51,
        "tradedShares": 1765,
        "amount": 1905692.7
    },
    "2024-10-20": {
        "price": {"open": 1058, "max": 1079, "min": 1040, "close": 1045, "prevClose": 1079, "diff": -34},
        "numTrans": 61,
        "tradedShares": 2758,
        "amount": 2900378.72
    }
}

# ✅ Process JSON data and store in 'documents'
documents = process_stock_json(json_data)

# ✅ Print the first document
if documents:
    print(documents[0])
else:
    print("No documents were created. Check your input data.")


page_content='Date: 2024-10-17
        Open Price: 1100
        Max Price: 1100
        Min Price: 1061
        Close Price: 1079
        Previous Close: 1080
        Price Difference: -1
        Number of Transactions: 51
        Traded Shares: 1765
        Total Trade Amount: 1905692.7' metadata={'date': '2024-10-17', 'open_price': 1100, 'close_price': 1079, 'max_price': 1100, 'min_price': 1061, 'num_transactions': 51, 'traded_shares': 1765}


In [16]:
print(type(all_json_data))  # Should be a dictionary, NOT a list
print(list(all_json_data.keys())[:5])  # Check the first few keys

stock_documents = process_stock_json(all_json_data)


<class 'list'>


AttributeError: 'list' object has no attribute 'keys'

In [33]:
from langchain.schema import Document
# Function to Split Text into Chunks
from langchain.text_splitter import CharacterTextSplitter
from langchain.schema import Document  # Correct import

def chunk_and_embed(text: str, chunk_size: int = 1000, chunk_overlap: int = 200):
    text_splitter = CharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separator="."
    )
    chunks = text_splitter.split_text(text)
    return [Document(page_content=chunk) for chunk in chunks]


In [34]:
all_json_data = []
for path in folder_paths:
	json_data = load_json_files(path)
	all_json_data.extend(json_data)

processed_text = process_json_data(all_json_data)
chunks = chunk_and_embed(processed_text)
